[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/biology_aging/blob/main/notebooks/build-cure-vs-slow-aging.ipynb)

# Curing One Disease, or Slowing Aging? 🎯
### A hands-on companion to Chapter 8

Chapter 8 asks the question the whole book has been building toward: should we fight the diseases of aging **one at a time**, or go after **aging itself**? Here you'll build a population that's dying of several age-driven diseases at once, then try both strategies and watch which one actually moves the needle. The result is the book's central argument — and it surprises almost everyone.

No setup. Runs in Colab. Press ▶ top to bottom.

## Step 0 — Tools, and how we'll model death

Each disease gets an **age-rising risk** that roughly *doubles every 8 years* — the real pattern of aging mortality, named after Benjamin Gompertz: `hazard(age) = A · exp(B · age)`. `A` sets how common it is; `B` how fast it accelerates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

age = np.linspace(0, 121, 12101); da = age[1]-age[0]

# each cause: (A, B) for hazard A*exp(B*age). Numbers are illustrative but realistic.
causes = {
    "heart disease":   (7e-6, 0.093),
    "cancer":          (7e-6, 0.091),
    "dementia":        (5e-8, 0.150),   # rare until very old, then steep
    "everything else": (1.1e-5, 0.092), # stroke, diabetes, kidney, infection...
}
print("Four competing causes loaded.")

## Part 1 — A population that ages

Let's look at the risk of each disease as people get older. (Note the log scale — these risks rise *enormously* with age.)

In [ ]:
plt.figure(figsize=(7,4.3))
for name,(A,B) in causes.items():
    plt.semilogy(age, A*np.exp(B*age), label=name)
plt.xlabel("age (years)"); plt.ylabel("yearly risk of dying from cause (log scale)")
plt.title("Every disease of aging rises steeply with age"); plt.legend(); plt.xlim(0,100)
plt.tight_layout(); plt.show()

## Part 2 — Who survives, and for how long

At every age you face **all** these risks at once, and whichever strikes first ends the story. Add the hazards, turn them into a survival curve, and read off **life expectancy** (the average age at death = the area under the survival curve).

In [ ]:
def survival_and_LE(causes, slow=0.0, cure=None):
    H = np.zeros_like(age)
    for name,(A,B) in causes.items():
        if name == cure: continue
        H += A*np.exp(B*(1-slow)*age)          # 'slow' stretches the aging clock
    S = np.exp(-np.cumsum(H)*da)               # fraction still alive at each age
    LE = np.sum((S[:-1]+S[1:])/2)*da           # area under survival curve
    return S, LE

S0, LE0 = survival_and_LE(causes)
print(f"Baseline life expectancy: {LE0:.1f} years")

plt.figure(figsize=(7,4.3))
plt.plot(age, 100*S0, color="#264653", lw=2)
plt.xlabel("age (years)"); plt.ylabel("% of people still alive")
plt.title(f"Survival curve - life expectancy {LE0:.0f} years"); plt.xlim(0,110)
plt.tight_layout(); plt.show()

## Part 3 — Strategy A: cure a disease outright

Now the best case for the one-disease-at-a-time approach: imagine we **completely cure cancer** — not treat it, *eliminate* it. How many years does that add?

In [ ]:
S_nocancer, LE_nocancer = survival_and_LE(causes, cure="cancer")
S_noheart,  LE_noheart  = survival_and_LE(causes, cure="heart disease")
S_nodem,    LE_nodem    = survival_and_LE(causes, cure="dementia")

print(f"Cure CANCER entirely     -> life expectancy {LE_nocancer:.1f}  (gain {LE_nocancer-LE0:+.1f} years)")
print(f"Cure HEART DISEASE       -> life expectancy {LE_noheart:.1f}  (gain {LE_noheart-LE0:+.1f} years)")
print(f"Cure DEMENTIA            -> life expectancy {LE_nodem:.1f}  (gain {LE_nodem-LE0:+.1f} years)")

plt.figure(figsize=(7,4.3))
plt.plot(age, 100*S0,        color="#264653", lw=2, label="today")
plt.plot(age, 100*S_nocancer,color="#e76f51", lw=2, ls="--", label="cancer cured")
plt.xlabel("age (years)"); plt.ylabel("% still alive")
plt.title("Curing cancer outright barely moves the curve"); plt.xlim(0,110); plt.legend()
plt.tight_layout(); plt.show()

Only a couple of years — for *eliminating* the disease. Why so little? Because the people you saved from cancer are now exposed to **heart disease and dementia, waiting right behind it.** That's **competing risks**: when several causes all rise with age, removing one just hands you to the others. It's why, in the real world, curing cancer is estimated to add only a few years to life expectancy.

## Part 4 — Strategy B: slow aging itself

Instead of removing one cause, what if we **slow the rate at which *all* the risks rise** — turn down everyone's biological clock a little? Let's compare.

In [ ]:
print(f"Cure cancer outright   : {LE_nocancer-LE0:+.1f} years")
for s in (0.10, 0.20, 0.30):
    _, LEs = survival_and_LE(causes, slow=s)
    print(f"Slow aging by {int(s*100)}%      : {LEs-LE0:+.1f} years")

S_slow, LE_slow = survival_and_LE(causes, slow=0.20)

plt.figure(figsize=(7,4.3))
plt.plot(age, 100*S0,         color="#264653", lw=2, label="today")
plt.plot(age, 100*S_nocancer, color="#e76f51", lw=2, ls="--", label="cancer cured")
plt.plot(age, 100*S_slow,     color="#2a9d8f", lw=2, label="aging slowed 20%")
plt.xlabel("age (years)"); plt.ylabel("% still alive")
plt.title("Slowing aging shifts the WHOLE curve"); plt.xlim(0,115); plt.legend()
plt.tight_layout(); plt.show()

plt.figure(figsize=(6.5,3.8))
plt.bar(["cure\ncancer","cure\nheart","slow aging\n20%"],
        [LE_nocancer-LE0, LE_noheart-LE0, LE_slow-LE0],
        color=["#e76f51","#e76f51","#2a9d8f"])
plt.ylabel("years of life expectancy gained"); plt.title("One disease vs the root")
plt.tight_layout(); plt.show()

There's the book's whole argument in one bar chart. Curing any *single* disease buys a few years; **slowing aging itself buys far more**, because the diseases of aging share a common root — pull the root and you push them all back at once.

And the deeper prize (Chapter 8's real point) isn't just *more* years — slowing aging pushes back the diseases *together*, so it compresses the sick years at the end rather than just extending them: more **healthy** life, not just more life.

> The honesty still holds. This is the *case for* targeting aging — it is not proof we can do it in humans. The model shows why the strategy is worth pursuing; whether any intervention truly slows human aging is exactly what remains unproven.

### Go further
- Add more causes (stroke, diabetes as their own curves) and watch curing one matter even less.
- Change the Gompertz doubling time `B`. What if aging accelerated faster, or slower?
- Model a *partial* cure (cut a hazard in half instead of to zero) and compare to a *partial* slowing of aging.